# Stage 03a — Aircraft Identity, Scope & Maturity

**What it does.** For every patent in a batch, joins the wizard's human record
(approval, aircraftName, duplicates) onto the PatSeer metadata, proposes a real
aircraft name, reads the energy source and the take-off mode out of the text
**with the sentence each came from**, and writes one workbook per batch:

    <data_matched>/<Batch_NN>/aircraft_identity_<Batch_NN>.xlsx

**Inputs:** `batches.xlsx`, the PatSeer export, `reference/evtol_gazetteer.csv`
and the live wizard exports in `03c_CORRECTED_wizard_exports`. No images.

## What you review (three decisions per patent)

| Decision | Machine columns you read | Where it says so | You type into |
|---|---|---|---|
| Real aircraft name | `aircraft_group` (never empty), `aircraft_name` (proposal), `aircraft_name_in_text` | `aircraft_name_section` / `_quote` | `aircraft_name_human` |
| Electric? | `is_electric`, `powertrain` | `powertrain_section` / `_quote` | `is_electric_human` = Yes / Hybrid / **No** (= disapproved) |
| VTOL? | `takeoff_mode` (VTOL / STOL / V/STOL / CTOL) | `takeoff_section` / `_quote` | `takeoff_human` = VTOL / **STOL** / CTOL (STOL = disapproved) |

## You only review what is still open

`review_queue` names the fields this row still needs (`name`, `electric`,
`takeoff`, or a combination) and is **empty for every row already answered**.
A field leaves the queue when something trustworthy settled it:

| Settled by | What it covers |
|---|---|
| the annotator | a patent they disapproved is out of the corpus and never queued; a reason that NAMES the thing ("Not VTOL", a future "Out of TD — Electric") decides that field outright |
| the patent's own words | a keyword powertrain (0.80) or a take-off phrase, both with the sentence quoted |
| the gazetteer | a curated company → aircraft powertrain (0.70-0.95) |
| the company | a company whose entire curated portfolio is battery-electric (`is_electric_source = company`) |
| you | anything you have already typed into a `*_human` cell |

A bare "Out of Domain / Out of TD" disapproval settles **nothing** — it is about
to be split into sub-reasons, so today it does not say which one applied.
A low-confidence SBERT powertrain settles nothing either: it stays visible in
the `powertrain` column with its score, but `is_electric` abstains rather than
letting a 0.30 similarity read as a verdict.

Then set `review_status` = `done` (or `skip`). The `*_final` columns are your
value when typed, else the machine value — they are what the next stage joins.
Re-running this notebook **keeps every `*_human` cell, `review_status`, `notes`**
and backs the old file up first.

`aircraft_group` is the working name, following the annotator's own scheme:
wizard `aircraftName` → inherited from the duplicate's original (D1/D2) →
original + letter (D3) → `<assignee> <N>` generated, `N` unique corpus-wide.
`aircraft_name` is only a **proposal**: a gazetteer name is inferred from company
+ year and `aircraft_name_in_text` tells you whether the patent ever says it.

**A note on quotes.** The name proposal, the take-off mode and a keyword-matched
powertrain always point at a real sentence. With `USE_SBERT = True` the rows the
keyword pass could not answer are filled by cosine similarity instead, and their
`*_section` says `sbert (no literal sentence)` — trust those less, and check the
patent before signing one off.

**Also on the sheet, not reviewed now:** geography (`assignee_country`,
`region`, `pub_office`), dates (`priority_year` < `app_year` < `pub_year`,
`snapshot_date`), legal stage from the export's *Legal Status Current* column,
citations, scope / architecture predictions, specs. `wizard_approved` lets you
restrict any statistic to the approved corpus.

**How to run:** set `BATCH_ID` below, run every cell top to bottom, open the
workbook, type your decisions, re-run cells 4–9 whenever you want fresh machine
columns (your cells survive). All logic lives in `src/` — `src/identity_pipeline.py`
is the runner, `src/wizard_link.py` the naming, `src/text_citation.py` the quotes.

In [ ]:
# ── 1. Configuration — the only knobs ──────────────────────────────────────
BATCH_ID = 1        # which Batch_NN sheet of batches.xlsx to process
LIMIT    = None     # e.g. 5 for a quick smoke test; None = the whole batch

# The PatSeer export this run describes. Ages and citation rates are measured
# to THIS date, not to today, and it is written to every row (snapshot_date).
SNAPSHOT_DATE = "2026-06-08"     # 1639__dataset_08_06_26.xlsx

# Where the wizard's human record is read from. None = cfg paths.corrected_wizard_exports.
WIZARD_DIR = None

# Read the full Description + Claims (not just title/abstract/claim/summary) for
# the powertrain keyword pass. Most patents never state their energy source in
# the abstract but do state it in the body, and a hit there still quotes a real
# sentence — it answered 432 of the 577 patents the short fields left open.
# Costs ~15 s and ~60 MB of text per run.
USE_FULL_TEXT = True

# PatentSBERTa fills powertrain / scope / innovation_field / industry for the
# rows the keyword pass leaves empty (~5 s to load, ~1.4 min per batch on the
# 2080 Ti). Those values are cosine-similarity judgements, NOT literal matches:
# their *_section column reads "sbert (no literal sentence)" because there is no
# quote to point at. The name, take-off and keyword-powertrain citations are
# unaffected. False = keyword / regex / gazetteer only, everything quotable.
USE_SBERT = True

# LLM step: "off" for the review pass. "export" writes questions into the workbook
# for you to paste into a chat (no API key); "api" calls Claude directly.
LLM_MODE  = "off"
LLM_MODEL = "claude-opus-5"
LLM_ONLY_UNRESOLVED = True   # only ask about patents the gazetteer could not name

In [ ]:
# ── 2. Setup: repo on sys.path, config, models ─────────────────────────────
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = next((c for c in [_cwd, *_cwd.parents]
                  if (c / "src").exists() and (c / "config.yaml").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")
for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

import pandas as pd
from src.config_loader import load_config
from src import identity_pipeline as pipeline
from src import aircraft_identity as ai

cfg = load_config()
print(f"repo_root : {repo_root}")

In [ ]:
# ── 3. Load PatentSBERTa (optional) ────────────────────────────────────────
# Same model and cache folder as 01a_wizard_feed, so the weights are not
# downloaded twice. A missing GPU or an offline machine degrades this stage to
# its keyword/regex/gazetteer passes rather than failing the run.
import os

sbert = None
if USE_SBERT:
    # Must be set BEFORE sentence_transformers imports huggingface_hub, which
    # freezes the cache path at import time (same note 01a carries).
    os.environ.setdefault("HF_HUB_CACHE", str(cfg["paths"]["sbert_cache"]))
    os.environ.setdefault("HF_HOME",      str(cfg["paths"]["sbert_cache"]))
    try:
        import torch
        from sentence_transformers import SentenceTransformer
        device = "cuda" if torch.cuda.is_available() else "cpu"
        sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa",
                                    cache_folder=str(cfg["paths"]["sbert_cache"]),
                                    device=device)
        print(f"PatentSBERTa loaded on {device}.")
    except Exception as exc:
        print(f"⚠  SBERT unavailable ({type(exc).__name__}: {exc}) — "
              f"running keyword/regex/gazetteer only.")
else:
    print("USE_SBERT = False — running keyword/regex/gazetteer only.")

In [ ]:
# ── 4. Read the batch, the PatSeer export, the gazetteer and the wizard record ─
inputs = pipeline.load_inputs(cfg, BATCH_ID, limit=LIMIT, repo_root=repo_root,
                              snapshot_date=SNAPSHOT_DATE, wizard_dir=WIZARD_DIR,
                              full_text=USE_FULL_TEXT)

In [ ]:
# ── 5. Run every signal and assemble the rows ──────────────────────────────
# Order inside: identity row -> scope/specificity -> maturity -> percentiles.
# See src/identity_pipeline.py for why that order is not interchangeable.
results = pipeline.analyse(inputs, sbert)

In [ ]:
# ── 6. Build the LLM questions (optional) ──────────────────────────────────
prompts, llm_answers = {}, {}

if LLM_MODE != "off":
    prompts = pipeline.build_prompts(inputs, results, only_unresolved=LLM_ONLY_UNRESOLVED)

if LLM_MODE == "api" and prompts:
    print(f"Calling {LLM_MODEL} for {len(prompts)} patent(s)...")
    llm_answers = ai.ask_claude(prompts, model=LLM_MODEL)
    results = pipeline.analyse(inputs, sbert, llm_answers=llm_answers)
elif LLM_MODE == "export":
    print(f"{len(prompts)} prompt(s) will be written to the LLM_Prompts sheet.\n"
          f"Paste each reply into its `llm_answer` cell, save, then run cell 8.\n"
          f"\nGive the chat this system prompt once, before the questions:\n")
    print(ai.LLM_SYSTEM_PROMPT)
else:
    print("LLM step skipped.")

In [ ]:
# ── 7. Write the workbook ──────────────────────────────────────────────────
# Backs up any existing file, then carries your hand edits forward: every
# *_human column, review_status, reviewed_by/at, notes and pasted llm_answer,
# plus any field whose *_source you set to "human". The *_final columns are
# recomputed from your entries. Safe to re-run over a sheet you are editing.
out_path = pipeline.export(inputs, results, prompts)

In [ ]:
# ── 8. Merge pasted LLM answers back in (run after filling the sheet) ──────
INGEST_LLM_ANSWERS = False   # flip to True once the llm_answer column is filled

if INGEST_LLM_ANSWERS:
    # The whole analysis is redone, not patched: an LLM-sourced name is evidence
    # the patent depicts the aircraft in a way a gazetteer name is not, so
    # specificity and aircraft_link can legitimately change once answers land.
    llm_answers = pipeline.read_pasted_answers(out_path)
    results = pipeline.analyse(inputs, sbert, llm_answers=llm_answers)
    pipeline.export(inputs, results, prompts)
else:
    print("INGEST_LLM_ANSWERS = False — nothing to ingest yet.")

In [ ]:
# ── 9. Coverage report + THE QUEUE — the only rows that still need you ──────
ident = pipeline.report(results)

# Everything else is already answered by something trustworthy: the annotator's
# own disapproval reason, a gazetteer or keyword powertrain read from the
# patent's words, or a company that builds nothing but electric aircraft.
queue = ident[ident["review_queue"].notna()]
print(f"\n{len(queue)} of {len(ident)} rows need you. Work down this table:")
queue[ai.REVIEW_COLUMNS]

In [ ]:
# ── 10. Every batch at once (optional) ─────────────────────────────────────
# One workbook per batch plus a combined table. The LLM step is skipped here on
# purpose — see run_all_batches() for why. Human edits in each existing workbook
# survive exactly as in cell 7.
RUN_ALL_BATCHES = False

if RUN_ALL_BATCHES:
    combined = pipeline.run_all_batches(cfg, sbert, repo_root=repo_root,
                                        snapshot_date=SNAPSHOT_DATE, wizard_dir=WIZARD_DIR,
                                        full_text=USE_FULL_TEXT)
else:
    print("RUN_ALL_BATCHES = False — single-batch run only.")

## Where this leaves you

The workbook is the deliverable. **Identity** joins onto anything keyed by
`patent_id` (the wizard export uses `Patent_ID`); **Figures** says what each
drawing depicts; **Evidence** is the audit trail behind every value; **README**
is the column dictionary, including the review contract.

### Reviewing

1. Sort or filter on `needs_review` / `review_reason` if you want the flagged
   rows first, but every row is yours to confirm.
2. For the name: `aircraft_group` is already a valid, unique working name. Type
   a real name into `aircraft_name_human` only when you know it — the proposal
   in `aircraft_name` is company-level unless `aircraft_name_in_text` = Yes.
3. For electric and VTOL: read the quote, type `No` / `STOL` to disapprove.
4. Set `review_status` = `done`. Save. Re-run cells 4–9 any time.

### Three rules for using it in the thesis

1. **Use the `*_final` columns and `wizard_approved == TRUE`.** The machine
   columns are proposals; the finals carry your decisions.
2. **Rank maturity on `forward_citations_per_year`**, never the raw count, and
   remember every age is measured to `snapshot_date`. Patents with a priority
   date within ~2 years of the snapshot are under-counted (publication lag), so
   truncate time series there.
3. **Report the coverage, don't hide it.** "We identified the real aircraft for
   N of M patents" is a result about drafting practice, not a failed run.

### Still open (decide before the join stage)

- Which year a time series should use — `priority_year`, `app_year` (filing) or
  `pub_year`. The sheet carries all three; this export has no grant date.
- How the finals are appended to the wizard export (long-format rows vs a
  separate sheet). Not needed for the review itself.